Installer les dépendances nécessaires :

- langchain (core)
- langchain-community (intégrations)
- pypdf (PDF)
- beautifulsoup4 + lxml (parsing HTML)
- requests (HTTP)
- arxiv (API arXiv)
- wikipedia (API Wikipedia)

> Note : on reste sur LangChain v0.2


In [ ]:
%pip install -U langchain langchain-community pypdf beautifulsoup4 lxml requests arxiv wikipedia


### 1. Text Loader (fichier .txt)
---

📌 Requirements :
- langchain-community

📁 Préparation :
- Crée un fichier `speech.txt` dans le même dossier que ton notebook.
- Mets-y du texte (quelques paragraphes).


In [3]:
from langchain_community.document_loaders import TextLoader

loader = TextLoader("speech.txt", encoding="utf-8")
text_documents = loader.load()

print("Nombre de documents :", len(text_documents))
print("Type :", type(text_documents[0]))
print("Contenu (extrait) :\n", text_documents[0].page_content[:300])
print("\nMetadata :", text_documents[0].metadata)

Nombre de documents : 1
Type : <class 'langchain_core.documents.base.Document'>
Contenu (extrait) :
 Dans la douce brise de Joal, un jeune garçon nommé Léopold écoutait les chants anciens de son peuple Sérère. 
Ces mélodies murmuraient des histoires de rois et de baobabs sacrés.
Des années plus tard, loin des rivages du Sénégal et sous le ciel gris de Paris, 
il transforma ces souvenirs en poésie. 

Metadata : {'source': 'speech.txt'}


### 2. PDF Loader (fichier .pdf)

-----

📌 Requirements :
- langchain-community
- pypdf

📁 Préparation :
- Mets un fichier `attention.pdf` dans le même dossier que ton notebook.
- Le loader va découper le PDF page par page (1 page = 1 Document).


In [4]:
from langchain_community.document_loaders import PyPDFLoader

loader = PyPDFLoader("attention.pdf")
docs = loader.load()

print("Nombre de pages chargées :", len(docs))
print("Type d'un élément :", type(docs[0]))
print("\n--- Extrait page 1 ---\n", docs[0].page_content[:400])
print("\nMetadata page 1 :", docs[0].metadata)

Nombre de pages chargées : 15
Type d'un élément : <class 'langchain_core.documents.base.Document'>

--- Extrait page 1 ---
 Provided proper attribution is provided, Google hereby grants permission to
reproduce the tables and figures in this paper solely for use in journalistic or
scholarly works.
Attention Is All You Need
Ashish Vaswani∗
Google Brain
avaswani@google.com
Noam Shazeer∗
Google Brain
noam@google.com
Niki Parmar∗
Google Research
nikip@google.com
Jakob Uszkoreit∗
Google Research
usz@google.com
Llion Jones∗
G

Metadata page 1 : {'producer': 'pdfTeX-1.40.25', 'creator': 'LaTeX with hyperref', 'creationdate': '2023-08-03T00:07:29+00:00', 'author': '', 'keywords': '', 'moddate': '2023-08-03T00:07:29+00:00', 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.25 (TeX Live 2023) kpathsea version 6.3.5', 'subject': '', 'title': '', 'trapped': '/False', 'source': 'attention.pdf', 'total_pages': 15, 'page': 0, 'page_label': '1'}


### 3. Web Loader (page web)

----

📌 Requirements :
- langchain-community
- beautifulsoup4
- lxml
- requests

🎯 Objectif :
Charger uniquement certaines parties utiles d’une page web (titre + contenu) via BeautifulSoup.

In [5]:
from langchain_community.document_loaders import WebBaseLoader
import bs4

url = "https://lilianweng.github.io/posts/2023-06-23-agent/"

loader = WebBaseLoader(
    web_paths=(url,),
    bs_kwargs=dict(
        parse_only=bs4.SoupStrainer(
            class_=("post-title", "post-content", "post-header")
        )
    )
)

web_docs = loader.load()

print("Nombre de documents web :", len(web_docs))
print("Extrait :\n", web_docs[0].page_content[:500])
print("\nMetadata :", web_docs[0].metadata)

USER_AGENT environment variable not set, consider setting it to identify your requests.


Nombre de documents web : 1
Extrait :
 

      LLM Powered Autonomous Agents
    
Date: June 23, 2023  |  Estimated Reading Time: 31 min  |  Author: Lilian Weng


Building agents with LLM (large language model) as its core controller is a cool concept. Several proof-of-concepts demos, such as AutoGPT, GPT-Engineer and BabyAGI, serve as inspiring examples. The potentiality of LLM extends beyond generating well-written copies, stories, essays and programs; it can be framed as a powerful general problem solver.
Agent System Overview#
In

Metadata : {'source': 'https://lilianweng.github.io/posts/2023-06-23-agent/'}


### 4. Arxiv Loader (papers scientifiques)
---

📌 Requirements :
- langchain-community
- arxiv

`Exemple` :
On récupère le papier "Attention is All You Need" via son identifiant : 1706.03762

In [ ]:
%pip install pymupdf

In [7]:
from langchain_community.document_loaders import ArxivLoader

docs_arxiv = ArxivLoader(query="2602.15019", load_max_docs=2).load()

print("Nombre de documents arXiv :", len(docs_arxiv))
print("Extrait :\n", docs_arxiv[0].page_content[:2000])
print("\nMetadata :", docs_arxiv[0].metadata)

Nombre de documents arXiv : 1
Extrait :
 Hunt Globally: Deep Research AI Agents for Drug Asset Scouting
in Investing, Business Development, and Search & Evaluation
Alisa Vinogradova1*, Vlad Vinogradov1*, Luba Greenwood1,3, Ilya Yasny1,2, Dmitry Kobyzev1,2,
Shoman Kasbekar, Kong Nguyen1, Dmitrii Radkevich1, Roman Doronin1, Andrey Doronichev1
1 Bioptic.io, San Francisco, CA
2 LanceBio Ventures
3 Harvard Business School
info@optic.inc
Abstract
Drug assets scouting is a time-consuming task for many in-
vestors and Business Development and Search & Evaluation
professionals in the biopharma field. Often, the data is al-
ready available on the public web and can be accessed by
modern Deep Research AI agents without the need to pur-
chase expensive proprietary software. However, general Deep
Research agents still cannot match the quality of human ex-
perts in identifying all potential drug assets that meet the
search criteria and fit complex queries. In this paper, we pro-
pose a robust benchm

### 5. Wikipedia Loader
--- 

📌 Requirements :
- langchain-community
- wikipedia

🎯 Objectif :
Récupérer une ou plusieurs pages Wikipedia sous forme de Documents.


In [ ]:
from langchain_community.document_loaders import WikipediaLoader

docs_wiki = WikipediaLoader(query="Generative AI", load_max_docs=2).load()

print("Nombre de documents Wikipedia :", len(docs_wiki))
print("Extrait :\n", docs_wiki[0].page_content[:500])
print("\nMetadata :", docs_wiki[0].metadata)

## Take Away

On a appris à charger des documents depuis :

- Texte (.txt) → TextLoader
- PDF → PyPDFLoader
- Web → WebBaseLoader
- arXiv → ArxivLoader
- Wikipedia → WikipediaLoader

➡️ Sortie standard : une liste d'objets `Document`
Chaque `Document` contient :
- `page_content` : le texte
- `metadata` : source, page, url, etc.